# 08 - Dimension x Questionnaire correlations per model (Real + Simulated)

Implements Phase D of `docs/revision/docs/analysis.md` per the
extended spec in `docs/revision/CORRELATION_DIM_QUEST_INSTRUCTIONS.md`.

**What this notebook computes:**
- Table 10a: per-model 2x5 dimension-questionnaire matrix, **two correlations per cell**:
  - `rho_sim`: LLM dim score vs LLM questionnaire score (internal consistency)
  - `rho_real`: LLM dim score vs real-patient questionnaire score (predictive validity)
- Table 10b: cross-model paired-bootstrap delta_rho anchored on GPT-5 (sim correlations).
- Table 10d: questionnaire inter-correlations across 5 variables (3 models + real).

Questionnaire variables used (5):
  - PCS total (0-52)
  - BPI Total mean (0-10)
  - BPI Intensity / Severity (mean of 4 pain-intensity items 'worst/least/average/right now', 0-10)
  - BPI Interference (mean of 7 daily-life interference items, 0-10)
  - TSK total (11-44)


## 1. Parameters

In [ ]:
MODELS_TO_COMPARE = [
    'gpt-5',
    'deepseek-r1',
    'claude-sonnet-4-5-thinking',
]
ANCHOR_MODEL = 'gpt-5'

DIMS = ['severidad_score', 'discapacidad_score']
QUESTS = [
    'pcs_total',
    'bpi_total_mean',
    'bpi_intensity_avg',     # BPI Intensity / Severity (mean of 4 pain-intensity items)
    'bpi_interference_avg',  # BPI Interference (mean of 7 daily-life interference items)
    'tsk_total',
]

BOOT_ITERS_CORR = 2000        # bootstrap iterations for per-cell CIs
BOOT_ITERS_DELTA = 5000       # bootstrap iterations for cross-model deltas
RNG_SEED = 42


## 2. Setup

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root / "src"))
from pain_narratives.analysis import revision_data_layer as dl

OUT_DIR = notebook_dir / 'outputs' / 'revision'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Models in scope: {MODELS_TO_COMPARE}')
print(f'Anchor: {ANCHOR_MODEL}')


## 3. Load model data and aggregate across runs

In [ ]:
def aggregate_runs(df, cols):
    return df.groupby('narrative_hash')[cols].mean().reset_index()

raw = {}
agg = {}
for tag in MODELS_TO_COMPARE:
    slug = tag.replace('-', '_')
    df = dl.load_processed(f'synth_{slug}.parquet')
    raw[tag] = df
    agg[tag] = aggregate_runs(df, DIMS + QUESTS)
    sev_na = agg[tag]['severidad_score'].isna().sum()
    dis_na = agg[tag]['discapacidad_score'].isna().sum()
    print(f'  {tag:<30} narratives={len(agg[tag]):>3}  sev_NaN={sev_na}  dis_NaN={dis_na}')


## 4. Load real-data reference

Real-patient questionnaire data on the same 152 narratives (joined by `narrative_hash`).
Renaming `_mean` -> `_avg` on the BPI subscale columns so real and synth share the same column names.


In [ ]:
real_152 = dl.load_processed('real_152.parquet')
real_quests = (real_152[['narrative_hash', 'pcs_total', 'bpi_total_mean',
                          'bpi_intensity_mean', 'bpi_interference_mean',
                          'tsk_total']]
                .rename(columns={'bpi_intensity_mean': 'bpi_intensity_avg',
                                 'bpi_interference_mean': 'bpi_interference_avg'}))
print(f'Real questionnaire reference: {len(real_quests)} narratives')
print(f'  columns: {list(real_quests.columns)}')
real_quests.describe()[QUESTS].round(2)


## 5. Helpers

In [ ]:
rng_master = np.random.default_rng(RNG_SEED)

def correlation_with_ci(x, y, n_boot=BOOT_ITERS_CORR, rng=None):
    '''Spearman rho + raw p + percentile bootstrap 95% CI on rho.'''
    rng = rng or np.random.default_rng(RNG_SEED)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 4:
        return {'rho': np.nan, 'p_raw': np.nan,
                'ci_lo': np.nan, 'ci_hi': np.nan, 'n': n}
    rho, p = spearmanr(x, y)
    idx = rng.integers(0, n, size=(n_boot, n))
    rx = pd.Series(x).rank(method='average').to_numpy()
    ry = pd.Series(y).rank(method='average').to_numpy()
    rx_b = rx[idx]
    ry_b = ry[idx]
    rx_b = rx_b - rx_b.mean(axis=1, keepdims=True)
    ry_b = ry_b - ry_b.mean(axis=1, keepdims=True)
    num = (rx_b * ry_b).sum(axis=1)
    den = np.sqrt((rx_b ** 2).sum(axis=1) * (ry_b ** 2).sum(axis=1))
    boots = np.where(den > 0, num / den, np.nan)
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return {'rho': float(rho), 'p_raw': float(p),
            'ci_lo': float(lo), 'ci_hi': float(hi), 'n': int(n)}

def signif_mark(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return ''


## 6. Table 10a - per-model dimension x questionnaire correlations (Sim + Real)

Each (model, dimension, questionnaire) row has two correlations:
- `rho_sim`: LLM dim score (mean across runs) vs LLM questionnaire score (mean across runs)
- `rho_real`: LLM dim score (mean across runs) vs real-patient questionnaire score

Both correlations use the same set of narratives (joined by `narrative_hash`).
FDR-BH applied separately to the sim and real p-value series within each model.


In [ ]:
rows = []
for tag, df in agg.items():
    # Join LLM aggregated values with real questionnaire values on narrative_hash
    merged = df.merge(
        real_quests.rename(columns={q: f'{q}__real' for q in QUESTS}),
        on='narrative_hash', how='inner',
    )
    for d in DIMS:
        for q in QUESTS:
            r_sim = correlation_with_ci(merged[d], merged[q])
            r_real = correlation_with_ci(merged[d], merged[f'{q}__real'])
            rows.append({
                'model': tag, 'dimension': d, 'questionnaire': q,
                'n_sim': r_sim['n'], 'rho_sim': r_sim['rho'], 'p_sim_raw': r_sim['p_raw'],
                'ci_lo_sim': r_sim['ci_lo'], 'ci_hi_sim': r_sim['ci_hi'],
                'n_real': r_real['n'], 'rho_real': r_real['rho'], 'p_real_raw': r_real['p_raw'],
                'ci_lo_real': r_real['ci_lo'], 'ci_hi_real': r_real['ci_hi'],
            })
df_10a = pd.DataFrame(rows)
# FDR-BH per model, separately on sim and real p-value series
for tag in MODELS_TO_COMPARE:
    mask = df_10a['model'] == tag
    for col_in, col_out in (('p_sim_raw', 'p_sim_fdr'), ('p_real_raw', 'p_real_fdr')):
        pvals = df_10a.loc[mask, col_in].fillna(1.0).to_numpy()
        _, p_fdr, _, _ = multipletests(pvals, method='fdr_bh')
        df_10a.loc[mask, col_out] = p_fdr
df_10a['signif_sim'] = df_10a['p_sim_fdr'].map(signif_mark)
df_10a['signif_real'] = df_10a['p_real_fdr'].map(signif_mark)
df_10a.to_csv(OUT_DIR / '08_table10a_per_model_correlations.csv', index=False)
# Print compact view
compact = df_10a[['model','dimension','questionnaire','n_sim','rho_sim','signif_sim',
                  'rho_real','signif_real']].copy()
print(compact.round(3).to_string(index=False))


## 7. Table 10b - cross-model paired bootstrap on rho_sim (anchored on GPT-5)

In [ ]:
def paired_bootstrap_delta(df_a, df_b, x_var, y_var,
                            n_boot=BOOT_ITERS_DELTA, rng=None):
    rng = rng or np.random.default_rng(RNG_SEED)
    merged = (df_a[['narrative_hash', x_var, y_var]]
              .merge(df_b[['narrative_hash', x_var, y_var]],
                     on='narrative_hash', suffixes=('_a', '_b'))
              .dropna())
    n = len(merged)
    if n < 4:
        return {'delta_rho': np.nan, 'ci_lo': np.nan, 'ci_hi': np.nan,
                'p_bootstrap': np.nan, 'n': n}
    xa = merged[f'{x_var}_a'].to_numpy()
    ya = merged[f'{y_var}_a'].to_numpy()
    xb = merged[f'{x_var}_b'].to_numpy()
    yb = merged[f'{y_var}_b'].to_numpy()
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        rho_a, _ = spearmanr(xa[idx], ya[idx])
        rho_b, _ = spearmanr(xb[idx], yb[idx])
        deltas[i] = rho_a - rho_b
    return {'delta_rho': float(np.nanmean(deltas)),
            'ci_lo': float(np.nanpercentile(deltas, 2.5)),
            'ci_hi': float(np.nanpercentile(deltas, 97.5)),
            'p_bootstrap': float(2.0 * min((deltas > 0).mean(), (deltas < 0).mean())),
            'n': int(n)}

rows = []
for other in MODELS_TO_COMPARE:
    if other == ANCHOR_MODEL:
        continue
    for d in DIMS:
        for q in QUESTS:
            res = paired_bootstrap_delta(agg[ANCHOR_MODEL], agg[other], d, q)
            rows.append({'model_a': ANCHOR_MODEL, 'model_b': other,
                          'dimension': d, 'questionnaire': q, **res})
df_10b = pd.DataFrame(rows)
df_10b.to_csv(OUT_DIR / '08_table10b_cross_model_delta.csv', index=False)
print(df_10b.round(3).to_string(index=False))


## 8. Table 10d - questionnaire inter-correlations across the 5 variables

All pairwise correlations among {PCS, BPI Total, BPI Intensity, BPI Interference, TSK} for
each of the 3 models (aggregated across runs) and for the real-patient data.
10 pairs x 4 sources = 40 rows.

In [ ]:
def quest_matrix(df, vars=QUESTS):
    rows = []
    for i, qi in enumerate(vars):
        for qj in vars[i + 1:]:
            res = correlation_with_ci(df[qi], df[qj])
            rows.append({'q1': qi, 'q2': qj, **res})
    return pd.DataFrame(rows)

blocks = []
for tag in MODELS_TO_COMPARE:
    mqm = quest_matrix(agg[tag])
    mqm.insert(0, 'source', tag)
    blocks.append(mqm)
rr = quest_matrix(real_quests)
rr.insert(0, 'source', 'real')
blocks.append(rr)
df_10d = pd.concat(blocks, ignore_index=True)
df_10d.to_csv(OUT_DIR / '08_table10d_quest_inter.csv', index=False)
print(df_10d.round(3).to_string(index=False))


## 9. Table 10c - questionnaire vs real, delta-rho per model

For each pair (q1, q2), compute the model's Spearman rho minus the real rho. Tells us
which model best preserves the real inter-questionnaire structure.


In [ ]:
real_qm = quest_matrix(real_quests)
delta_rows = []
for tag in MODELS_TO_COMPARE:
    mqm = quest_matrix(agg[tag])
    mqm['source'] = tag
    merged = mqm.merge(real_qm[['q1', 'q2', 'rho']].rename(columns={'rho': 'rho_real'}),
                       on=['q1', 'q2'])
    merged['delta'] = merged['rho'] - merged['rho_real']
    delta_rows.append(merged)
df_10c = pd.concat(delta_rows, ignore_index=True)
df_10c.to_csv(OUT_DIR / '08_table10c_quest_vs_real.csv', index=False)
print(df_10c[['source','q1','q2','rho','rho_real','delta']].round(3).to_string(index=False))
print()
print('Mean |delta vs real| per model:')
print(df_10c.assign(abs_delta=df_10c['delta'].abs())
             .groupby('source')['abs_delta'].mean().round(3).to_string())


## 10. Figure 1 - per-model heatmap grid (Sim correlations)

In [ ]:
fig, axes = plt.subplots(1, len(MODELS_TO_COMPARE),
                         figsize=(5 * len(MODELS_TO_COMPARE), 3.5),
                         sharey=True, squeeze=False)
for ax, tag in zip(axes[0], MODELS_TO_COMPARE):
    pivot = (df_10a.query('model == @tag')
             .pivot(index='dimension', columns='questionnaire', values='rho_sim'))
    pivot = pivot[QUESTS]
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, cbar=(ax is axes[0][-1]),
                square=False, linewidths=0.3)
    ax.set_title(f'{tag} (sim)')
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
fig.suptitle('LLM dim vs LLM questionnaire (Spearman rho)')
fig.tight_layout()
fig.savefig(FIG_DIR / '08_fig1_per_model_heatmap_sim.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved fig1 (sim).')


## 11. Figure 1b - per-model heatmap grid (Real correlations)

In [ ]:
fig, axes = plt.subplots(1, len(MODELS_TO_COMPARE),
                         figsize=(5 * len(MODELS_TO_COMPARE), 3.5),
                         sharey=True, squeeze=False)
for ax, tag in zip(axes[0], MODELS_TO_COMPARE):
    pivot = (df_10a.query('model == @tag')
             .pivot(index='dimension', columns='questionnaire', values='rho_real'))
    pivot = pivot[QUESTS]
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, cbar=(ax is axes[0][-1]),
                square=False, linewidths=0.3)
    ax.set_title(f'{tag} (real)')
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
fig.suptitle('LLM dim vs REAL questionnaire (Spearman rho)')
fig.tight_layout()
fig.savefig(FIG_DIR / '08_fig1b_per_model_heatmap_real.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved fig1b (real).')


## 12. Figure 2 - side-by-side Sim vs Real bars per (model, dim, quest)

In [ ]:
plot_df = df_10a.copy()
plot_df['cell'] = (plot_df['dimension'].str.replace('_score', '').str.title()
                   + ' x ' + plot_df['questionnaire'])
cells_order = [f"{d.replace('_score','').title()} x {q}" for d in DIMS for q in QUESTS]

fig, axes = plt.subplots(len(MODELS_TO_COMPARE), 1,
                         figsize=(14, 4 * len(MODELS_TO_COMPARE)),
                         squeeze=False)
for ax, tag in zip(axes[:, 0], MODELS_TO_COMPARE):
    sub = plot_df.query('model == @tag').set_index('cell').reindex(cells_order)
    x = np.arange(len(sub))
    bar_w = 0.4
    err_sim = [sub['rho_sim'] - sub['ci_lo_sim'], sub['ci_hi_sim'] - sub['rho_sim']]
    err_real = [sub['rho_real'] - sub['ci_lo_real'], sub['ci_hi_real'] - sub['rho_real']]
    ax.bar(x - bar_w / 2, sub['rho_sim'], bar_w, label='sim (LLM vs LLM)',
           yerr=err_sim, capsize=2, color='#4C72B0')
    ax.bar(x + bar_w / 2, sub['rho_real'], bar_w, label='real (LLM vs patient)',
           yerr=err_real, capsize=2, color='#DD8452')
    ax.set_xticks(x)
    ax.set_xticklabels(cells_order, rotation=30, ha='right')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel('Spearman rho')
    ax.set_title(f'{tag}: dimension x questionnaire correlations (mean +/- 95% bootstrap CI)')
    ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(FIG_DIR / '08_fig2_sim_vs_real_bars.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved fig2 (Sim vs Real bars).')


## 13. Figure 3 - Q x Q delta-vs-real forest plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
labels, y_pos, vals, errs_lo, errs_hi = [], [], [], [], []
row_count = 0
for tag in MODELS_TO_COMPARE:
    sub = df_10c.query('source == @tag').reset_index(drop=True)
    for _, r in sub.iterrows():
        labels.append(f'{tag} | {r.q1} vs {r.q2}')
        y_pos.append(row_count)
        vals.append(r['delta'])
        errs_lo.append(r['delta'] - (r['ci_lo'] - r['rho_real']))
        errs_hi.append((r['ci_hi'] - r['rho_real']) - r['delta'])
        row_count += 1
ax.errorbar(vals, y_pos, xerr=[errs_lo, errs_hi], fmt='o', capsize=3)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('delta rho (model - real)')
ax.set_title('Model questionnaire inter-correlations vs real reference')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(FIG_DIR / '08_fig3_delta_vs_real_forest.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved fig3.')


## 14. Interpretation summary

In [ ]:
mean_abs_delta_sim_cross = df_10b['delta_rho'].abs().mean()
mean_abs_delta_real_per_model = (df_10c.assign(abs_d=df_10c['delta'].abs())
                                  .groupby('source')['abs_d'].mean().round(3))
n_sig_cross = (df_10b['p_bootstrap'] < 0.05).sum()
n_total_cross = len(df_10b)

# Average rho_sim vs rho_real per model (signal of predictive validity gap)
summary = (df_10a.groupby('model')[['rho_sim', 'rho_real']]
           .agg(['mean', 'min', 'max']).round(3))

print('=' * 80)
print('INTERPRETATION SUMMARY')
print('=' * 80)
print(f'Cross-model delta_rho (sim, Table 10b): mean |delta| = {mean_abs_delta_sim_cross:.3f}')
print(f'  bootstrapped p<0.05 in {n_sig_cross}/{n_total_cross} cells')
print()
print('Per-model rho_sim vs rho_real (across 10 dim x quest cells):')
print(summary.to_string())
print()
print('Mean |delta vs real| per model on questionnaire inter-correlations:')
print(mean_abs_delta_real_per_model.to_string())
